<a href="https://colab.research.google.com/github/agcbls49/THESIS/blob/main/Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **INTRODUCTION**
This colab file contains the model training and model comparisons of YOLOv5n, YOLOv8n, YOLOv11n and YOLOv26n using a 70% train, 20% validation and 10% test data split. For more information, you can visit this Github [repository](https://github.com/agcbls49/THESIS).

<br>

---

# **SPECIFICATIONS**

The dataset is split into **70% train, 20% valid and 10% test** and every model is trained using the following default YOLO specifications:
1. 100 Epochs
2. All images resized to 640x640
3. 16 Batches

# **SETUP**

### Check if the correct runtime is set

The runtime should be A100 if the file is run in Google Colab. This is run to ensure that the runtime used is not CPU or a lower powered GPU in Colab.

In [ ]:
!nvidia-smi

Fri Oct  9 04:10:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Mounting the Dataset from Google Drive

This step mounts the Google Drive where the dataset is stored and copies the dataset.zip file to Google Colab's storage allowing the dataset to be used for model training.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Copy dataset.zip from Drive to Colab's local storage
!cp "/content/gdrive/MyDrive/1_THESIS_DATASET/dataset1.zip" /content/

### Unzipping the dataset contents
This step simply unzips the content of dataset.zip in Google Colab's storage.

In [2]:
# Unzip the dataset folder contents
!unzip -q /content/dataset1.zip -d /content/dataset

### Installing Ultralytics for YOLO
This step installs the ultralytics library which is needed for training YOLO models.

In [ ]:
# Install ultralytics for YOLO
!pip install ultralytics -q

### Creating a YAML file
This step creates a YAML file which is needed for training YOLO models. It contains the path of the unzipped dataset, where the train, validation and test images are located, and the names of the classes along with their class numbers.

In [ ]:
import yaml
import os

data = {
    'path': '/content/dataset/dataset1',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 24,
    'names': [
        'backpack',
        'book',
        'cup',
        'glasses',
        'headset',
        'key',
        'phone',
        'remote',
        'wallet',
        'water bottle',
        'body spray',
        'card',
        'charger',
        'comb',
        'flashlight',
        'glasses case',
        'medicine',
        'nail clipper',
        'shoe',
        'watch',
        'body scrub',
        'cane',
        'lighter',
        'towel'
    ]
}

yaml_path = '/content/data.yaml'

# write file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("written")

# verify file exists
print("exists:", os.path.exists(yaml_path))

# verify size
print("size:", os.path.getsize(yaml_path))

# print contents
with open(yaml_path, 'r') as f:
    content = f.read()

print("CONTENT:")
print(content)

### **(OPTIONAL)** Deleting Conflicting Cache

**This step is done only if the YOLO commands produce a YAML error otherwise this can be skipped**. <br><br> The code below deletes the cached YAML file ensuring that the code runs the proper yaml file.

In [ ]:
!find /content/dataset/dataset -name "*.cache" -delete

# **RUNNING THE MODELS**

### **YOLO v5 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolov5n.pt epochs=100 imgsz=640 batch=16 name=yolov5n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolov5n_run/weights/best.pt split=test name=yolov5n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLOv5n summary (fused): 84 layers, 2,507,624 parameters, 0 gradients, 7.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2791.2±1663.8 MB/s, size: 3754.9 KB)
val: Scanning /content/dataset/dataset1/labels/test... 793 images, 73 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 1.2Kit/s 0.6s
val: New cache created: /content/dataset/dataset1/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 4.2it/s 11.8s
                   all        793       1432      0.902      0.804      0.875      0.702
              backpack         39         39      0.879      0.872      0.948      0.803
                  book         50        128      0.791      0.695      0.785      0.606
                   cup         83        169      0.863      0.747       0.79      0.587
               glasses         53         53    

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolov5n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolov5n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolov5n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolov5n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolov5n_test_results.txt")



---



### **YOLO v8 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolov8n.pt epochs=100 imgsz=640 batch=16 name=yolov8n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [ ]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolov8n_run/weights/best.pt split=test name=yolov8n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
Model summary (fused): 72 layers, 3,010,328 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1686.2±178.4 MB/s, size: 2887.6 KB)
val: Scanning /content/dataset/dataset1/labels/test.cache... 793 images, 73 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 144.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.8it/s 13.0s
                   all        793       1432      0.907      0.839      0.894      0.728
              backpack         39         39      0.974      0.968      0.988      0.879
                  book         50        128      0.821      0.766       0.83      0.652
                   cup         83        169      0.853      0.823      0.842      0.637
               glasses         53         53      0.937      0.906       0.93        0.8
               headset

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolov8n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolov8n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolov8n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolov8n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolov8n_test_results.txt")



---



### **YOLO v11 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11n.pt epochs=100 imgsz=640 batch=16 name=yolo11n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [6]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolo11n_run/weights/best.pt split=test name=yolo11n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11n summary (fused): 100 layers, 2,586,832 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1682.5±1067.0 MB/s, size: 2911.6 KB)
val: Scanning /content/dataset/dataset1/labels/test... 793 images, 73 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 1.2Kit/s 0.6s
val: New cache created: /content/dataset/dataset1/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 3.8it/s 13.2s
                   all        793       1432      0.906      0.845      0.899      0.738
              backpack         39         39      0.935      0.974      0.973      0.864
                  book         50        128      0.899      0.694      0.854      0.683
                   cup         83        169      0.878      0.806       0.82      0.622
               glasses         53         53   

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolo11n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolo11n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolo11n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolo11n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolo11n_test_results.txt")



---



### **YOLO v26 NANO MODEL**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo26n.pt epochs=100 imgsz=640 batch=16 name=yolo26n_run

**Run the trained model against the test images not included in the code above and print the final performance metrics.**

In [9]:
!yolo detect val data=/content/data.yaml model=runs/detect/yolo26n_run/weights/best.pt split=test name=yolo26n_test

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26n summary (fused): 120 layers, 2,379,516 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2148.2±606.6 MB/s, size: 1335.0 KB)
val: Scanning /content/dataset/dataset1/labels/test.cache... 793 images, 73 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 793/793 103.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 4.2it/s 12.0s
                   all        793       1432      0.915      0.844      0.895      0.751
              backpack         39         39      0.945      0.923      0.944      0.858
                  book         50        128      0.828      0.727      0.839      0.684
                   cup         83        169      0.878      0.822      0.829      0.649
               glasses         53         53       0.96      0.904      0.939      0.799
               head

In [ ]:
from ultralytics import YOLO

# Export best.pt to TensorFlow Lite
model = YOLO("runs/detect/yolo26n_run/weights/best.pt")
model.export(format="tflite")

# Zip the ENTIRE runs/detect folder (this now includes train + val + test results)
!zip -r yolo26n_run.zip runs/detect

# Download the ZIP
from google.colab import files
files.download("yolo26n_run.zip")

metrics = model.val(data="/content/data.yaml", split="test")
with open("yolo26n_test_results.txt", "w") as f:
    f.write(f"Precision: {metrics.box.mp}\n")
    f.write(f"Recall: {metrics.box.mr}\n")
    f.write(f"mAP50: {metrics.box.map50}\n")
    f.write(f"mAP50-95: {metrics.box.map}\n")
files.download("yolo26n_test_results.txt")



---



**After comparing the results from both data split configurations** *(the other data split is in the [Experiment 2 colab file](https://colab.research.google.com/drive/1ki3rl4tZwEHgTyHqFQJyDXnauPp2_YqN?usp=sharing)),* **the selected model will be further evaluated using 150 training epochs.**

<br>

This 150 epoch experiment will be on the [Experiment 3 colab file](https://colab.research.google.com/drive/1AE36BZFs3d6x4c7-419_y96wG4L6LFC7?usp=sharing).